# Google Colab Auto-AOI Generator

Upload two files from the app:

- the study video
- `colab-aoi-job-*.json`

This notebook generates editable object-edge polygon AOIs for the browser app:

1. Florence-2 detects candidate objects and regions from the exported prompts.
2. SAM 2 segments each Florence-2 detection box into an object mask.
3. OpenCV extracts the largest mask contour.
4. `approxPolyDP` simplifies that contour into an editable polygon.
5. The notebook downloads AOI JSON that can be imported with **Import Colab AOIs** or **Load AOI JSON** in the app.


In [ ]:
!pip -q install transformers accelerate opencv-python pillow supervision
!pip -q install git+https://github.com/facebookresearch/sam2.git

import json
import math
import re
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import torch
from google.colab import files
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from transformers import AutoModelForCausalLM, AutoProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
FLORENCE_MODEL_ID = 'microsoft/Florence-2-base'
SAM2_MODEL_ID = 'facebook/sam2.1-hiera-small'
SAM2_CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt'
SAM2_CHECKPOINT_PATH = Path('sam2.1_hiera_small.pt')
SAM2_CONFIG_CANDIDATES = [
    'configs/sam2.1/sam2.1_hiera_s.yaml',
    'sam2.1/sam2.1_hiera_s.yaml',
]

print('Using', DEVICE)


In [ ]:
uploaded = files.upload()
paths = [Path(name) for name in uploaded.keys()]
job_path = next(path for path in paths if path.suffix.lower() == '.json')
video_path = next(path for path in paths if path != job_path)

job = json.loads(job_path.read_text())
assert job.get('kind') == 'aoi-colab-job', 'Upload a colab-aoi-job JSON exported from the app.'

video_meta = job.get('video', {})
policy = job.get('aoiPolicy', {})

prompt_values = policy.get('prompts', [])
if isinstance(prompt_values, str):
    prompt_values = re.split(r'[\n,]+', prompt_values)
prompts = [str(prompt).strip().lower() for prompt in prompt_values if str(prompt).strip()]

def policy_number(name, default):
    try:
        value = float(policy.get(name, default))
    except (TypeError, ValueError):
        return default
    return value if math.isfinite(value) else default

sample_interval = max(0.25, policy_number('sampleIntervalSec', 1.0))
detector_model = policy.get('detectorModel') or FLORENCE_MODEL_ID
segmenter_model = SAM2_MODEL_ID
requested_segmenter_model = policy.get('segmenterModel') or SAM2_MODEL_ID
max_polygon_points = int(min(240, max(12, round(policy_number('maxPolygonPoints', 80)))))
polygon_epsilon_ratio = min(0.02, max(0.001, policy_number('polygonSimplificationEpsilon', 0.003)))
analysis_padding_px = int(min(128, max(0, round(policy_number('analysisPaddingPx', 18)))))

print('Video:', video_path)
print('Projection:', video_meta.get('projection', 'equirectangular'))
print('Stereo:', video_meta.get('stereoLayout', 'mono'))
print('Prompts:', prompts)
print('Sample interval:', sample_interval)
print('Detector:', detector_model)
print('Segmenter:', segmenter_model)
if requested_segmenter_model != segmenter_model:
    print('Requested segmenter is', requested_segmenter_model, '- using SAM 2.1 small for free Colab compatibility.')
print('Max polygon points:', max_polygon_points)
print('Polygon epsilon ratio:', polygon_epsilon_ratio)
print('Analysis padding px:', analysis_padding_px)


In [ ]:
processor = AutoProcessor.from_pretrained(detector_model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    detector_model,
    trust_remote_code=True,
    torch_dtype=DTYPE,
).to(DEVICE)
model.eval()


def ensure_sam2_checkpoint():
    if SAM2_CHECKPOINT_PATH.exists():
        return
    print('Downloading SAM 2.1 small checkpoint...')
    urllib.request.urlretrieve(SAM2_CHECKPOINT_URL, SAM2_CHECKPOINT_PATH)


def load_sam2_predictor():
    ensure_sam2_checkpoint()
    device = DEVICE
    checkpoint = str(SAM2_CHECKPOINT_PATH)
    last_error = None
    for model_cfg in SAM2_CONFIG_CANDIDATES:
        try:
            sam2_model = build_sam2(model_cfg, checkpoint, device=device)
            print('SAM 2 config:', model_cfg)
            return SAM2ImagePredictor(sam2_model), model_cfg
        except Exception as exc:
            last_error = exc
    raise RuntimeError('Unable to load SAM 2.1 small config from installed package paths.') from last_error


predictor, sam2_config = load_sam2_predictor()


def clamp(value, lower, upper):
    return max(lower, min(upper, value))


def normalize_yaw(value):
    normalized = ((value + 180) % 360) - 180
    return 180 if normalized == -180 and value > 0 else normalized


def slug(label):
    text = re.sub(r'[^a-z0-9]+', '-', label.lower()).strip('-')
    return text or 'generated-aoi'


def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    x1, y1 = max(ax1, bx1), max(ay1, by1)
    x2, y2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    denom = area_a + area_b - inter
    return inter / denom if denom else 0


def prompt_allowed(label):
    if not prompts:
        return True
    lowered = label.lower()
    return any(prompt in lowered or lowered in prompt for prompt in prompts)


def crop_stereo_eye(frame, stereo_layout, projection):
    if projection == 'flat':
        return frame
    height, width = frame.shape[:2]
    if stereo_layout == 'side-by-side':
        return frame[:, :width // 2]
    if stereo_layout == 'top-bottom':
        return frame[:height // 2, :]
    return frame


def sample_frames(video_file, interval_sec, stereo_layout, projection):
    cap = cv2.VideoCapture(str(video_file))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if frame_count else 0
    times = []
    t = 0.0
    while t <= max(duration, 0.01):
        times.append(round(t, 3))
        t += max(0.25, interval_sec)
    frames = []
    for t in times:
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, frame = cap.read()
        if not ok:
            continue
        frame = crop_stereo_eye(frame, stereo_layout, projection)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((t, Image.fromarray(rgb)))
    cap.release()
    if projection == 'flat':
        width, height = source_width, source_height
    elif stereo_layout == 'side-by-side':
        width, height = source_width // 2, source_height
    elif stereo_layout == 'top-bottom':
        width, height = source_width, source_height // 2
    else:
        width, height = source_width, source_height
    return frames, {
        'width': width,
        'height': height,
        'sourceWidth': source_width,
        'sourceHeight': source_height,
        'durationSec': duration,
        'eye': 'full' if projection == 'flat' else 'left',
    }


def detect_frame(image):
    task = '<OD>'
    inputs = processor(text=task, images=image, return_tensors='pt').to(DEVICE, DTYPE)
    with torch.inference_mode():
        generated_ids = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=1024,
            num_beams=3,
        )
    text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(text, task=task, image_size=image.size)
    result = parsed.get(task, {})
    labels = result.get('labels', [])
    boxes = result.get('bboxes', [])
    return [
        {'label': label, 'bbox': [float(v) for v in box]}
        for label, box in zip(labels, boxes)
        if prompt_allowed(label)
    ]


def set_sam_image(image):
    predictor.set_image(np.asarray(image.convert('RGB')))


def segment_box(bbox, image_size):
    width, height = image_size
    x1, y1, x2, y2 = [float(value) for value in bbox]
    x1 = clamp(x1, 0, width)
    x2 = clamp(x2, 0, width)
    y1 = clamp(y1, 0, height)
    y2 = clamp(y2, 0, height)
    if x2 <= x1 or y2 <= y1:
        return None, 0.0

    box = np.array([x1, y1, x2, y2], dtype=np.float32)
    masks, scores, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=box[None, :],
        multimask_output=True,
    )
    masks = np.asarray(masks)
    scores = np.asarray(scores).reshape(-1)
    if masks.size == 0 or scores.size == 0:
        return None, 0.0
    if masks.ndim == 4:
        masks = masks.reshape((-1, masks.shape[-2], masks.shape[-1]))
    best_index = int(np.argmax(scores))
    return masks[best_index].astype(bool), float(scores[best_index])


def polygon_area(points):
    if len(points) < 3:
        return 0.0
    area = 0.0
    for index, (x1, y1) in enumerate(points):
        x2, y2 = points[(index + 1) % len(points)]
        area += (float(x1) * float(y2)) - (float(x2) * float(y1))
    return abs(area) / 2


def has_polygon_area(points, min_area=1.0):
    return len(points) >= 3 and polygon_area(points) > min_area


def mask_to_polygon(mask, max_points=80, epsilon_ratio=0.003):
    if mask is None:
        return []
    mask_array = np.asarray(mask)
    if mask_array.ndim > 2:
        mask_array = np.squeeze(mask_array)
    mask_u8 = (mask_array > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return []

    contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contour) <= 0:
        return []

    epsilon = max(0.0, float(epsilon_ratio)) * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    points = [(float(x), float(y)) for x, y in approx.reshape(-1, 2)]
    if not has_polygon_area(points):
        return []

    point_budget = max(3, int(max_points))
    if len(points) > point_budget:
        indices = np.linspace(0, len(points) - 1, num=point_budget, dtype=int)
        points = [points[index] for index in indices]
        if not has_polygon_area(points):
            return []
    return points


def normalize_polygon(points, width, height, projection):
    normalized = []
    safe_width = max(1, float(width))
    safe_height = max(1, float(height))
    for x, y in points:
        x_norm = clamp(float(x) / safe_width, 0, 1)
        y_norm = clamp(float(y) / safe_height, 0, 1)
        if projection == 'flat':
            normalized.append({
                'x': round(x_norm, 6),
                'y': round(y_norm, 6),
            })
        else:
            normalized.append({
                'yaw': round(normalize_yaw(x_norm * 360 - 180), 6),
                'pitch': round(90 - y_norm * 180, 6),
            })
    return normalized


def normalize_analysis_padding(padding_px, width, height, projection):
    safe_width = max(1, float(width))
    safe_height = max(1, float(height))
    safe_padding = max(0, float(padding_px))
    if projection == 'flat':
        return round(safe_padding / min(safe_width, safe_height), 6)
    return round(max(
        (safe_padding / safe_width) * 360,
        (safe_padding / safe_height) * 180,
    ), 6)


def group_detections(detections, iou_threshold=0.25):
    tracks = []
    for detection in detections:
        best_track = None
        best_score = 0
        for track in tracks:
            if track['label'] != detection['label']:
                continue
            score = iou(track['last_bbox'], detection['bbox'])
            if score > best_score:
                best_score = score
                best_track = track
        if best_track and best_score >= iou_threshold:
            best_track['detections'].append(detection)
            best_track['last_bbox'] = detection['bbox']
        else:
            tracks.append({
                'label': detection['label'],
                'last_bbox': detection['bbox'],
                'detections': [detection],
            })
    return tracks


In [ ]:
stereo_layout = video_meta.get('stereoLayout', 'mono')
projection = video_meta.get('projection', 'equirectangular')
frames, video_stats = sample_frames(video_path, sample_interval, stereo_layout, projection)
all_detections = []
analysis_padding = normalize_analysis_padding(
    analysis_padding_px,
    video_stats['width'],
    video_stats['height'],
    projection,
)

for index, (t, image) in enumerate(frames, start=1):
    frame_candidates = detect_frame(image)
    frame_detections = []
    width, height = image.size
    if frame_candidates:
        set_sam_image(image)
    for detection in frame_candidates:
        mask, mask_score = segment_box(detection['bbox'], image.size)
        polygon = mask_to_polygon(
            mask,
            max_points=max_polygon_points,
            epsilon_ratio=polygon_epsilon_ratio,
        )
        if len(polygon) < 3:
            continue
        points = normalize_polygon(polygon, width, height, projection)
        if len(points) < 3:
            continue
        frame_detections.append({
            **detection,
            't': t,
            'shape': 'polygon',
            'points': points,
            'maskScore': mask_score,
            'contourPoints': len(polygon),
        })
    all_detections.extend(frame_detections)
    print(
        f'{index}/{len(frames)} t={t:.2f}s '
        f'candidates={len(frame_candidates)} polygons={len(frame_detections)}'
    )

colors = ['#ffd166', '#5dd7c8', '#ff8a5c', '#8bd66f', '#ff4f9a', '#9fb7ff']
tracks = group_detections(all_detections)
aois = []
used_ids = set()

for idx, track in enumerate(tracks):
    if len(track['detections']) < 1:
        continue
    base_id = slug(track['label'])
    aoi_id = base_id
    suffix = 2
    while aoi_id in used_ids:
        aoi_id = f'{base_id}-{suffix}'
        suffix += 1
    used_ids.add(aoi_id)
    keyframes = [
        {
            't': det['t'],
            'points': det['points'],
            'maskScore': round(det['maskScore'], 6),
        }
        for det in track['detections']
    ]
    keyframes.sort(key=lambda keyframe: keyframe['t'])
    first = keyframes[0]
    aois.append({
        'id': aoi_id,
        'label': track['label'],
        'color': colors[idx % len(colors)],
        'space': 'video' if projection == 'flat' else 'panorama',
        'shape': 'polygon',
        'points': first['points'],
        'analysisPaddingPx': analysis_padding_px,
        'analysisPadding': analysis_padding,
        'generated': {
            'method': 'google-colab-florence2-sam2',
            'detectorModel': detector_model,
            'segmenterModel': segmenter_model,
            'sam2Config': sam2_config,
            'sampleIntervalSec': sample_interval,
            'projection': projection,
            'stereoLayout': stereo_layout,
            'eye': video_stats.get('eye', 'left'),
            'frameDetections': len(track['detections']),
            'maxPolygonPoints': max_polygon_points,
            'polygonSimplificationEpsilon': polygon_epsilon_ratio,
        },
        'keyframes': keyframes,
    })

output = {
    'video': {
        **video_meta,
        'width': video_stats['width'],
        'height': video_stats['height'],
        'sourceWidth': video_stats.get('sourceWidth'),
        'sourceHeight': video_stats.get('sourceHeight'),
        'eye': video_stats.get('eye', 'left'),
        'durationSec': round(video_stats['durationSec'], 3),
    },
    'aois': aois,
    'generatedBy': 'google-colab-florence2-sam2',
    'sourceJob': job,
}

out_path = Path('generated-colab-aois.json')
out_path.write_text(json.dumps(output, indent=2))
print(f'Generated {len(aois)} polygon AOIs')
files.download(str(out_path))
